# Advanced Analytics Notebook

Comprehensive time series analysis with multiple forecasting models and ensemble methods.

In [ ]:
# Cell 1: Import all necessary modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
import warnings
from datetime import datetime, timedelta
import sys

# Set up warnings
warnings.filterwarnings('ignore')

# Configure matplotlib and seaborn
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Import project modules
from src.models.arima import ARIMAModel
from src.models.prophet import ProphetModel
from src.models.exponential_smoothing import ExponentialSmoothingModel
from src.models.ensemble import ModelEnsemble, compare_models
from src.visualization.plotter import TimeSeriesPlotter
from src.utils.metrics import calculate_mae, calculate_rmse, calculate_mape

print("All modules imported successfully!")

In [ ]:
# Cell 2: Load data from processed CSV or use synthetic data
import os

data_path = 'data/processed/validated_combined.csv'

if os.path.exists(data_path):
    # Load from file
    print(f"Loading data from {data_path}...")
    df = pd.read_csv(data_path)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    print(f"Loaded {len(df)} records from {df['timestamp'].min()} to {df['timestamp'].max()}")
else:
    # Generate synthetic data for demonstration
    print("Data file not found. Generating synthetic data...")
    np.random.seed(42)
    dates = pd.date_range(start='2024-01-01', end='2024-12-31', freq='D')
    
    # Create synthetic alert counts with trend and seasonality
    trend = np.linspace(100, 200, len(dates))
    seasonality = 50 * np.sin(np.arange(len(dates)) * 2 * np.pi / 365)
    noise = np.random.normal(0, 20, len(dates))
    values = trend + seasonality + noise
    values = np.maximum(values, 0)  # Ensure non-negative
    
    df = pd.DataFrame({
        'timestamp': dates,
        'value': values,
        'oblast': 'Synthetic',
        'source': 'synthetic'
    })
    
    print(f"Generated {len(df)} synthetic records")

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())

In [ ]:
# Cell 3: Aggregate to daily counts and visualize historical data
# If data has multiple records per day, aggregate them
if 'timestamp' in df.columns:
    df['date'] = pd.to_datetime(df['timestamp']).dt.date
    daily_counts = df.groupby('date').size().reset_index(name='value')
    daily_counts['timestamp'] = pd.to_datetime(daily_counts['date'])
    daily_counts = daily_counts[['timestamp', 'value']].sort_values('timestamp')
else:
    daily_counts = df[['timestamp', 'value']].sort_values('timestamp')

print(f"Daily aggregated data: {len(daily_counts)} days")
print(f"Date range: {daily_counts['timestamp'].min()} to {daily_counts['timestamp'].max()}")
print(f"\nDaily statistics:")
print(daily_counts['value'].describe())

# Visualize historical data
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_counts['timestamp'], daily_counts['value'], linewidth=2, marker='o', markersize=3, label='Daily Alerts')
ax.fill_between(daily_counts['timestamp'], daily_counts['value'], alpha=0.3)
ax.set_xlabel('Date')
ax.set_ylabel('Number of Alerts')
ax.set_title('Historical Air Raid Alert Data')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Historical data visualization complete.")

In [ ]:
# Cell 4: Train-test split (last 30 days for testing)
test_days = 30
split_idx = len(daily_counts) - test_days

train_data = daily_counts.iloc[:split_idx].copy()
test_data = daily_counts.iloc[split_idx:].copy()

print(f"Training set: {len(train_data)} days (from {train_data['timestamp'].min()} to {train_data['timestamp'].max()})")
print(f"Test set: {len(test_data)} days (from {test_data['timestamp'].min()} to {test_data['timestamp'].max()})")
print(f"\nTrain statistics:")
print(train_data['value'].describe())
print(f"\nTest statistics:")
print(test_data['value'].describe())

# Visualize train-test split
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train_data['timestamp'], train_data['value'], label='Training', linewidth=2, marker='o', markersize=3)
ax.plot(test_data['timestamp'], test_data['value'], label='Test', linewidth=2, marker='o', markersize=3, color='red')
ax.axvline(x=test_data['timestamp'].iloc[0], color='gray', linestyle='--', alpha=0.7, label='Split Point')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Alerts')
ax.set_title('Train-Test Split (Last 30 Days for Testing)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Train-test split complete.")

In [ ]:
# Cell 5: Build ensemble with all available models
# Create instances of all forecasting models
print("Initializing forecasting models...")

# ARIMA Model
arima_model = ARIMAModel(order=(1, 1, 1))
print("✓ ARIMA(1,1,1) initialized")

# Prophet Model
prophet_model = ProphetModel(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95
)
print("✓ Prophet (with yearly & weekly seasonality) initialized")

# Exponential Smoothing Model
exp_smoothing_model = ExponentialSmoothingModel(
    seasonal_periods=30,
    trend='add',
    seasonal='add'
)
print("✓ ExponentialSmoothing (seasonal_periods=30) initialized")

# Create Ensemble
ensemble = ModelEnsemble()
ensemble.add_model('ARIMA', arima_model)
ensemble.add_model('Prophet', prophet_model)
ensemble.add_model('ExponentialSmoothing', exp_smoothing_model)

print(f"\n✓ Ensemble created with {len(ensemble.models)} models")
print(f"Models in ensemble: {list(ensemble.models.keys())}")

In [ ]:
# Cell 6: Fit all models on training data
print("Training all models on training data...\n")

try:
    ensemble.fit(train_data)
    print("✓ All models trained successfully!")
except Exception as e:
    print(f"⚠️ Error during training: {e}")
    print("Continuing with individual model fitting...")
    
    for name, model in ensemble.models.items():
        try:
            model.fit(train_data)
            print(f"  ✓ {name} trained successfully")
        except Exception as e:
            print(f"  ✗ {name} failed: {e}")

print(f"\nTraining complete. Ensemble is ready for forecasting.")

In [ ]:
# Cell 7: Generate 7-day forecasts from each model
print("Generating 7-day forecasts from each model...\n")

forecast_steps = 7
forecasts = {}

for name, model in ensemble.models.items():
    try:
        forecast = model.forecast(steps=forecast_steps)
        forecasts[name] = forecast
        print(f"✓ {name}: {forecast}")
    except Exception as e:
        print(f"✗ {name} forecast failed: {e}")

print(f"\nForecast generation complete.")
print(f"Successfully generated forecasts from {len(forecasts)} models")

In [ ]:
# Cell 8: Visualize forecast comparison using plotter
print("Creating forecast comparison visualization...\n")

plotter = TimeSeriesPlotter()

# Create a combined dataset for visualization
# Use test data as reference
forecast_dates = pd.date_range(start=test_data['timestamp'].iloc[-1] + timedelta(days=1), periods=forecast_steps, freq='D')

# Prepare data for plotting
plot_df = daily_counts.copy()
plot_df = plot_df.rename(columns={'timestamp': 'ds', 'value': 'y'})

# Generate and visualize comparison
fig = plotter.plot_model_comparison(
    df=plot_df,
    forecasts=forecasts,
    timestamp_col='ds',
    value_col='y',
    title='7-Day Forecast Comparison: All Models',
    figsize=(16, 8)
)

plt.tight_layout()
plt.show()

print("✓ Forecast comparison visualization complete.")

In [ ]:
# Cell 9: Compare models quantitatively on test set using compare_models()
print("Comparing models quantitatively on test set...\n")

try:
    models_list = list(ensemble.models.values())
    comparison_results = compare_models(models_list, train_data, test_data, steps=7)
    
    comparison_df = pd.DataFrame(comparison_results)
    print("Model Comparison Results:")
    print(comparison_df.to_string(index=False))
    print(f"\n✓ Comparison complete.")
except Exception as e:
    print(f"⚠️ Comparison failed: {e}")
    print("Generating manual comparison...")
    
    # Manual comparison
    test_values = test_data['value'].values
    comparison_df = pd.DataFrame({
        'Model': list(forecasts.keys()),
        'Forecast_Mean': [np.mean(forecast) if isinstance(forecast, np.ndarray) else np.mean(forecast) for forecast in forecasts.values()],
        'Test_Mean': [test_values.mean()] * len(forecasts)
    })
    print(comparison_df.to_string(index=False))

In [ ]:
# Cell 10: Visualize metrics comparison using plotter
print("Creating metrics comparison visualization...\n")

try:
    # Create metrics visualization
    fig = plotter.plot_metrics_comparison(
        models_metrics=comparison_results,
        metric='MAE',
        figsize=(12, 6)
    )
    plt.show()
except Exception as e:
    print(f"⚠️ Metrics visualization failed: {e}")
    print("Creating alternative metrics visualization...")
    
    # Alternative visualization
    if len(comparison_df) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        x_pos = np.arange(len(comparison_df))
        ax.bar(x_pos, comparison_df['Forecast_Mean'].values, alpha=0.7, label='Forecast Mean')
        ax.axhline(y=comparison_df['Test_Mean'].iloc[0], color='red', linestyle='--', label='Test Mean')
        ax.set_xlabel('Model')
        ax.set_ylabel('Mean Value')
        ax.set_title('Model Forecast Means vs Test Mean')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(comparison_df['Model'].values)
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.show()
        print("✓ Alternative metrics visualization complete.")

In [ ]:
# Cell 11: Generate ensemble forecast (average of all models)
print("Generating ensemble forecast...\n")

try:
    ensemble_forecast = ensemble.ensemble_forecast(steps=7, method='mean')
    print(f"✓ Ensemble forecast (mean of all models): {ensemble_forecast}")
except Exception as e:
    print(f"⚠️ Ensemble forecast failed: {e}")
    print("Calculating manual ensemble forecast...")
    
    # Manual ensemble forecast
    forecast_arrays = []
    for forecast in forecasts.values():
        if isinstance(forecast, np.ndarray):
            forecast_arrays.append(forecast)
        else:
            forecast_arrays.append(np.array(forecast))
    
    if forecast_arrays:
        ensemble_forecast = np.mean(forecast_arrays, axis=0)
        print(f"✓ Ensemble forecast (mean of {len(forecast_arrays)} models): {ensemble_forecast}")

# Visualize ensemble forecast
fig, ax = plt.subplots(figsize=(14, 6))

# Plot historical data
ax.plot(daily_counts['timestamp'], daily_counts['value'], label='Historical Data', linewidth=2, marker='o', markersize=3)

# Plot ensemble forecast
forecast_dates = pd.date_range(start=daily_counts['timestamp'].iloc[-1] + timedelta(days=1), periods=7, freq='D')
ax.plot(forecast_dates, ensemble_forecast, label='Ensemble Forecast', linewidth=2.5, marker='s', markersize=6, color='red')

# Formatting
ax.axvline(x=daily_counts['timestamp'].iloc[-1], color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('Number of Alerts')
ax.set_title('Ensemble Forecast: Average of All Models')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Ensemble forecast visualization complete.")

In [ ]:
# Cell 12: Summary and recommendations
print("="*80)
print("ADVANCED ANALYTICS SUMMARY AND RECOMMENDATIONS")
print("="*80)

print("\n1. DATA OVERVIEW")
print(f"   - Total records analyzed: {len(daily_counts)} days")
print(f"   - Date range: {daily_counts['timestamp'].min().date()} to {daily_counts['timestamp'].max().date()}")
print(f"   - Training set: {len(train_data)} days")
print(f"   - Test set: {len(test_data)} days")
print(f"   - Mean daily alerts: {daily_counts['value'].mean():.2f}")
print(f"   - Std deviation: {daily_counts['value'].std():.2f}")

print("\n2. MODELS TRAINED")
for name in ensemble.models.keys():
    print(f"   ✓ {name}")

print("\n3. FORECAST RESULTS")
print(f"   Ensemble forecast (next 7 days):")
for i, (date, value) in enumerate(zip(forecast_dates, ensemble_forecast), 1):
    print(f"     Day {i} ({date.date()}): {value:.1f} alerts")

print("\n4. KEY INSIGHTS")
if len(daily_counts) > 0:
    trend = daily_counts['value'].iloc[-7:].mean() - daily_counts['value'].iloc[-14:-7].mean()
    if trend > 0:
        print(f"   - Upward trend detected: +{trend:.1f} alerts/day (last 7 days vs previous 7)")
    elif trend < 0:
        print(f"   - Downward trend detected: {trend:.1f} alerts/day (last 7 days vs previous 7)")
    else:
        print(f"   - Stable trend: minimal change between recent periods")

max_day = daily_counts.loc[daily_counts['value'].idxmax()]
    print(f"   - Peak alert day: {max_day['timestamp'].date()} with {max_day['value']:.0f} alerts")
    
    recent_days = daily_counts['value'].iloc[-7:]
    high_days = (recent_days > recent_days.mean()).sum()
    print(f"   - Recent volatility: {high_days} of last 7 days above average")

print("\n5. RECOMMENDATIONS")
print("   ✓ Ensemble method provides robust forecasting by combining multiple models")
print("   ✓ Monitor forecast vs actual values to detect model drift")
print("   ✓ Consider retraining models weekly with new data")
print("   ✓ Use individual model forecasts for uncertainty quantification")
print("   ✓ Investigate outliers for external events affecting alert patterns")

print("\n" + "="*80)
print("Analysis complete. Ready for production deployment or further investigation.")
print("="*80)